In [1]:
import pandas as pd

data = {
    "Context": [
        "the capital of france is paris",
        "the largest planet in the solar system is jupiter",
        "water freezes at zero degrees celsius",
        "the fastest land animal is cheetah",
        "python is a popular programming language"
    ],
    "Question": [
        "what is the capital of france",
        "which is the largest planet",
        "at what temperature does water freeze",
        "what is the fastest land animal",
        "what is python"
    ],
    "Answer": [
        "paris",
        "jupiter",
        "zero degrees celsius",
        "cheetah",
        "programming language"
    ]
}

df = pd.DataFrame(data)
df.to_csv("qa_dataset.csv", index=False)

print("Dataset created successfully: qa_dataset.csv")
print(df.head())


Dataset created successfully: qa_dataset.csv
                                             Context  \
0                     the capital of france is paris   
1  the largest planet in the solar system is jupiter   
2              water freezes at zero degrees celsius   
3                 the fastest land animal is cheetah   
4           python is a popular programming language   

                                Question                Answer  
0          what is the capital of france                 paris  
1            which is the largest planet               jupiter  
2  at what temperature does water freeze  zero degrees celsius  
3        what is the fastest land animal               cheetah  
4                         what is python  programming language  


In [15]:
import pandas as pd
import numpy as np

import tensorflow as tf

from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense,Input,Embedding
from tensorflow.keras.layers import TextVectorization
from tensorflow.keras.layers import MultiHeadAttention,LayerNormalization


from sklearn.preprocessing import LabelEncoder

In [4]:
df = pd.read_csv("qa_dataset.csv")


In [6]:
df["Input"] = df["Context"] + " " + df["Question"]

texts = df["Input"].values
answers = df["Answer"].values

In [7]:
# Encode answers (classification)
le = LabelEncoder()
y = le.fit_transform(answers)

In [8]:
num_classes = len(set(y))

# Text vectorization
vectorizer = TextVectorization(
    max_tokens=1000,
    output_mode="int",
    output_sequence_length=20
)

vectorizer.adapt(texts)

In [9]:
X = vectorizer(texts)

vocab_size = len(vectorizer.get_vocabulary())

# Transformer parameters
embed_dim = 64
num_heads = 2
ff_dim = 128

# Input layer
inputs = Input(shape=(20,))

# Embedding
x = Embedding(vocab_size, embed_dim)(inputs)

In [10]:
# Self-attention
attention_output = MultiHeadAttention(
    num_heads=num_heads,
    key_dim=embed_dim
)(x, x)

# Add & Normalize
x = LayerNormalization()(x + attention_output)

In [17]:
# Feed-forward
x = Dense(ff_dim, activation="relu")(x)
x = Dense(embed_dim)(x)

# Global pooling
x = tf.reduce_mean(x, axis=1)

# Output layer
outputs = Dense(num_classes, activation="softmax")(x)

ValueError: A KerasTensor cannot be used as input to a TensorFlow function. A KerasTensor is a symbolic placeholder for a shape and dtype, used when constructing Keras Functional models or Keras Functions. You can only use it as input to a Keras layer or a Keras operation (from the namespaces `keras.layers` and `keras.operations`). You are likely doing something like:

```
x = Input(...)
...
tf_fn(x)  # Invalid.
```

What you should do instead is wrap `tf_fn` in a layer:

```
class MyLayer(Layer):
    def call(self, x):
        return tf_fn(x)

x = MyLayer()(x)
```


In [19]:
# =====================================
# Question Answering using Transformer
# =====================================

import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras.layers import TextVectorization
from tensorflow.keras.layers import Input, Dense, Embedding
from tensorflow.keras.layers import MultiHeadAttention, LayerNormalization
from tensorflow.keras.models import Model
from sklearn.preprocessing import LabelEncoder

# Load dataset
df = pd.read_csv("qa_dataset.csv")

# Combine context + question
df["Input"] = df["Context"] + " " + df["Question"]

texts = df["Input"].values
answers = df["Answer"].values

# Encode answers (classification)
le = LabelEncoder()
y = le.fit_transform(answers)

num_classes = len(set(y))

# Text vectorization
vectorizer = TextVectorization(
    max_tokens=1000,
    output_mode="int",
    output_sequence_length=20
)

vectorizer.adapt(texts)

X = vectorizer(texts)

vocab_size = len(vectorizer.get_vocabulary())

# Transformer parameters
embed_dim = 64
num_heads = 2
ff_dim = 128

# Input layer
inputs = Input(shape=(20,))

# Embedding
x = Embedding(vocab_size, embed_dim)(inputs)

# Self-attention
attention_output = MultiHeadAttention(
    num_heads=num_heads,
    key_dim=embed_dim
)(x, x)

# Add & Normalize
x = LayerNormalization()(x + attention_output)

# Feed-forward
x = Dense(ff_dim, activation="relu")(x)
x = Dense(embed_dim)(x)

# Global pooling
from tensorflow.keras.layers import GlobalAveragePooling1D

# Feed Forward
x = Dense(ff_dim, activation="relu")(x)
x = Dense(embed_dim)(x)

# Global pooling (FIXED)
x = GlobalAveragePooling1D()(x)

# Output layer
outputs = Dense(num_classes, activation="softmax")(x)

# Output layer
outputs = Dense(num_classes, activation="softmax")(x)

# Build model
model = Model(inputs, outputs)

# Compile
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# Train
model.fit(
    X,
    y,
    epochs=50,
    verbose=1
)

print("\nTransformer QA Model Trained Successfully")


Epoch 1/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 6s 6s/step - accuracy: 0.2000 - loss: 1.6478
Epoch 2/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 79ms/step - accuracy: 0.8000 - loss: 1.4956
Epoch 3/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 83ms/step - accuracy: 1.0000 - loss: 1.3870
Epoch 4/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 83ms/step - accuracy: 1.0000 - loss: 1.2831
Epoch 5/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 78ms/step - accuracy: 1.0000 - loss: 1.1735
Epoch 6/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 78ms/step - accuracy: 1.0000 - loss: 1.0599
Epoch 7/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 85ms/step - accuracy: 1.0000 - loss: 0.9384
Epoch 8/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 162ms/step - accuracy: 1.0000 - loss: 0.8081
Epoch 9/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 77ms/step - accuracy: 1.0000 - loss: 0.6815
Epoch 10/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 80ms/step - accuracy: 1.0000 - loss: 0.5585
Epoch 11/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 173ms/step - accuracy: 1.0000 - loss: 0.4387
Epoch 12/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/step - accuracy: 1.0000 - loss: 0.3326
E

In [ ]:
def answer_question(context, question):

    combined = context + " " + question
    vector = vectorizer([combined])

    prediction = model.predict(vector, verbose=0)
    predicted_index = np.argmax(prediction)

    return le.inverse_transform([predicted_index])[0]

# Test
context = "the capital of france is paris"
question = "what is the capital of france"

print("Answer:", answer_question(context, question))


Answer: paris


: 